# Day 3 · Prompt design & iteration

*Day 3 — Linguistic Data Analysis II*

### How to use this notebook

This is your **single submission for the day**. It has two parts:

- **Part A · Tutorial** — improve a prompt through zero-shot → few-shot → chain-of-thought on the SAME CEFR-SP task, comparing macro-F1 at each step.
- **Part B · Corpus Lab** — run your own prompt-iteration study and error analysis (to be written).

You only edit the cells marked **✏️ YOU EDIT**. Cells marked **🔧 Library cell** are pre-written — run them, don't change them.

➡️ Work top to bottom. When you're done, **Runtime → Run all**, then **File → Download → Download `.ipynb`** and submit that file.

## Part A · Tutorial — three ways to prompt

Same pipeline as Day 2, same CEFR-SP data — only the **prompt** changes. We compare three techniques and watch macro-F1 move:

| Iteration | Technique | Idea |
|---|---|---|
| 0 | **zero-shot** | just describe the task |
| 1 | **few-shot** | add a few labeled examples |
| 2 | **chain-of-thought** | ask the model to reason before answering |

Record the macro-F1 (the `macro avg` row of `evaluate`) after each run so you can tell a story about what helped.

::: {.callout-important}
## From today you run the model yourself — you need a free API key
Days 1–2 used Colab's built-in Gemini (or a frozen file). From Day 3 on you call the model live and need your prompt runs to be **reproducible** (`temperature=0` + a fixed seed), so the notebook switches to the **Gemini API**. Get a free key and add it to Colab **Secrets** as `GEMINI_API_KEY` — one-time, ~2 minutes, no install. Full steps: [Get a free Gemini API key](../resources/tools/gemini-api-key.md). When the setup cell prints `LLM backend: Gemini API (...)` you're set; if it still says `Colab Gemini`, your secret isn't set or its notebook-access toggle is off. Don't worry about rate limits crashing your loop — that's already handled, and explained right after Setup, below.
:::

### Setup — run this first

In [ ]:
#@title 📦 Setup — run me first { display-mode: "form" }
# Imports + the LLM backend. No pip install needed in Colab.
import json, re, time, urllib.request, os
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd, seaborn as sns, matplotlib.pyplot as plt

MODEL_ID = "gemini-3.1-flash-lite"   # pinned model for the reproducible (API) backend

def _resolve_gemini_key():
    """Find a Gemini API key: Colab Secrets first (not auto-exported to env), then env."""
    try:
        from google.colab import userdata      # only exists in Colab
        key = userdata.get("GEMINI_API_KEY")
        if key:
            return key
    except Exception:
        pass                                    # not in Colab, or secret not set
    return os.environ.get("GEMINI_API_KEY")

def _make_api_backend(key):
    """Reproducible backend: Gemini API with temperature=0 + a fixed seed."""
    from google import genai
    from google.genai import types
    client = genai.Client(api_key=key)
    cfg = types.GenerateContentConfig(temperature=0, seed=42)
    return (lambda p: client.models.generate_content(
                model=MODEL_ID, contents=p, config=cfg).text,
            f"Gemini API ({MODEL_ID}, temperature=0, seed=42)")

# ---- keeps us from calling the model faster than the free tier allows ----------
# (explained step by step in Day 3, right after this cell — the short version:
#  wait a bit between calls, and if we still get told to slow down, wait longer
#  and try again, unless the message says we are out of quota for the whole day.)
def _looks_like_rate_limit(error):
    text = str(error).lower()
    return any(s in text for s in
               ["429", "resource_exhausted", "rate limit", "quota", "too many requests"])

def _looks_like_daily_quota(error):
    text = str(error).lower()
    return "per day" in text or "perday" in text.replace(" ", "")

def _suggested_delay(error, fallback):
    m = re.search(r"retry in ([0-9]+(?:\.[0-9]+)?)s", str(error).lower())
    return float(m.group(1)) + 1.0 if m else fallback

_last_call_time = 0.0   # generate_text remembers & updates this with `global`

def generate_text(prompt, max_retries=5):
    global _last_call_time
    for attempt in range(max_retries + 1):
        wait = _min_interval - (time.monotonic() - _last_call_time)
        if wait > 0:
            time.sleep(wait)
        try:
            _last_call_time = time.monotonic()
            return _raw_generate_text(prompt)
        except Exception as error:
            if not _looks_like_rate_limit(error):
                raise                                   # a real bug — don't hide it
            if _looks_like_daily_quota(error):
                raise RuntimeError(
                    "Daily quota used up for today — waiting won't help until it "
                    "resets. Come back tomorrow, or ask your instructor.") from error
            if attempt == max_retries:
                raise
            print(f"  (rate limited — waiting before trying again, attempt {attempt+1})")
            time.sleep(_suggested_delay(error, _min_interval * (attempt + 2)))
    raise RuntimeError("Still rate-limited after several tries.")

# Prefer the API key when set (reproducible); else fall back to colab.ai (demo).
_key = _resolve_gemini_key()
if _key:
    _raw_generate_text, _backend = _make_api_backend(_key)
    _min_interval = 4.4     # keeps us under gemini-3.1-flash-lite's 15-requests/minute cap
else:
    try:
        from google.colab import ai            # Colab's built-in Gemini — no key
        _raw_generate_text, _backend = (lambda p: ai.generate_text(p)), "Colab Gemini (demo, non-reproducible)"
        _min_interval = 13.2   # colab.ai publishes no rate limit — pace conservatively
    except ImportError:
        raise RuntimeError(
            "No LLM backend found. Run this notebook in Google Colab (free built-in "
            "Gemini, no key needed), or set GEMINI_API_KEY — in Colab via the Secrets "
            "panel, or as an environment variable when running locally. "
            "See resources/tools/gemini-api-key.md.")

# CEFR-SP gold set (72 sentences, 12 per level), fetched from the course repo.
GOLD_URL = "https://raw.githubusercontent.com/egumasa/linguistic-data-analysis-II-2026/main/sources/resources/datasets/gold/cefr_sentences.json"
LEVELS = ["A1", "A2", "B1", "B2", "C1", "C2"]
print(f"Setup done. LLM backend: {_backend}. scikit-learn ready.")

### Why your calls don't crash the lab — two different clocks

The free tier limits you in **two independent ways**, on two different clocks:

- **RPM** — requests per *minute*. A speed limit: how fast you're allowed to call.
- **RPD** — requests per *day*. A fuel tank: how much you're allowed to use, total, today.

A plain `for` loop over 72 sentences can blow through the RPM speed limit in the first few seconds — long before it's used any meaningful share of the day's fuel tank. This isn't hypothetical: while building this course, a loop tripped a 15-per-minute cap after only 16 calls in one minute, while just 126 of that day's 500-call budget had been used. The fix isn't "use less" — it's "go slower, and know which kind of limit you hit."

Your Setup cell above already has a guard built in that does exactly this. The rest of this section walks through it, piece by piece — you don't need it to keep working, but it's worth understanding what's protecting your Corpus Lab.

### Piece 1 — always leave a gap between calls (pacing)

The simplest fix: never call the model faster than the speed limit allows. If the limit is 15 calls per minute, that's one call every `60 / 15 = 4` seconds — so before each call, check how long it's been since the *last* one, and wait out the difference.

That means the function has to **remember** when the last call happened, across calls. Normally a function forgets everything the moment it returns — its local variables disappear. The `global` keyword is how you tell Python "no, remember this one, and share it across every call." That's the only new keyword in this whole guard.

Try it below — no model, no internet, just pacing:

In [ ]:
import time

_demo_last_call = 0.0        # remembered BETWEEN calls, thanks to `global`
DEMO_INTERVAL = 2            # seconds (the real guard uses 4.4s or 13.2s)

def wait_your_turn():
    global _demo_last_call
    wait = DEMO_INTERVAL - (time.monotonic() - _demo_last_call)
    if wait > 0:
        print(f"  waiting {wait:.1f}s so we don't call too often...")
        time.sleep(wait)
    _demo_last_call = time.monotonic()
    print("  → calling now!")

for i in range(3):
    wait_your_turn()

### Piece 2 — if you still get told to slow down, wait and try again

Pacing alone isn't quite enough — the server can still say "too fast, try again later," and a network hiccup can raise an error for reasons that have nothing to do with rate limits at all. `try`/`except` is Python's way of saying: *try this, and if it breaks, do something else instead of crashing.*

The something-else here depends on **what kind of failure it is**, which we can tell from the error message:

- **Not rate-limit shaped at all** (a typo in your prompt, a dropped connection) — a real bug. Don't retry it; let it raise so you notice.
- **Per-minute limit** — wait a bit and try again; the limit refills every minute, so this is temporary.
- **Per-day limit** — retrying is pointless. It won't refill until tomorrow no matter how long you wait, so the guard gives up immediately with a clear message instead of silently hanging.

A small demo — no model, just the `try`/`except` shape, retrying until it works:

In [ ]:
attempt_count = 0

def flaky_call():
    """Pretends to fail twice, then succeeds — like a real rate-limited call."""
    global attempt_count
    attempt_count += 1
    if attempt_count <= 2:
        raise Exception("429 rate limit — please slow down")
    return "success!"

for attempt in range(3):
    try:
        result = flaky_call()
        print("Got:", result)
        break
    except Exception as error:
        print(f"  attempt {attempt+1} failed ({error}) — trying again...")

### Putting the two pieces together

"Always wait a bit" (piece 1) plus "if told to slow down, wait longer and try again, but give up right away on a *daily* limit" (piece 2) is **exactly** what's inside `generate_text` in the Setup cell above. You've been calling it since your very first prompt on Day 1, and it's been quietly protecting every call — it'll protect every Corpus Lab loop you write from here on, too.

If you're curious to see the fancier version — one that also **remembers past answers so you never pay for the same prompt twice** — see [`resources/extra/handling-rate-limits.ipynb`](../resources/extra/handling-rate-limits.ipynb). It uses one more advanced idea we haven't covered yet (a function that builds and returns another function), but the underlying logic is identical to what you just read.

In [ ]:
#@title 🔧 Library cell: load_gold(url_or_path) → gold { display-mode: "form" }
def load_gold(url_or_path):
    """Read the canonical gold JSON: [{'id','text','label'}, ...]."""
    if str(url_or_path).startswith("http"):
        raw = urllib.request.urlopen(url_or_path).read().decode("utf-8")
        gold = json.loads(raw)
    else:
        gold = json.loads(open(url_or_path, encoding="utf-8").read())
    print(f"Loaded {len(gold)} items. First one:", gold[0])
    return gold

In [ ]:
#@title 🔧 Library cell: run_prompt(prompt, gold) → predictions { display-mode: "form" }
def _extract_level(text):
    """Pull the first A1/A2/B1/B2/C1/C2 out of the model's reply."""
    m = re.search(r"\b([ABC][12])\b", str(text).upper())
    return m.group(1) if m else "??"

def run_prompt(prompt, gold):
    """Send each item's `text` to the LLM via {text}, collect predicted labels."""
    predictions = []
    for i, item in enumerate(gold, 1):
        reply = generate_text(prompt.format(text=item["text"]))
        predictions.append(_extract_level(reply))
        if i % 12 == 0:
            print(f"  ...{i}/{len(gold)} done")
    print(f"Got {len(predictions)} predictions.")
    return predictions

In [ ]:
#@title 🔧 Library cell: evaluate(gold, predictions) → P/R/F1 + confusion matrix { display-mode: "form" }
def evaluate(gold, predictions):
    y_true = [item["label"] for item in gold]
    y_pred = predictions
    print(classification_report(y_true, y_pred, labels=LEVELS, zero_division=0))
    cm = confusion_matrix(y_true, y_pred, labels=LEVELS)
    plt.figure(figsize=(5.5, 4.5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=LEVELS, yticklabels=LEVELS)
    plt.xlabel("Predicted"); plt.ylabel("Gold"); plt.title("Confusion matrix")
    plt.tight_layout(); plt.show()

In [ ]:
#@title 🔧 Library cell: show_errors(gold, predictions) → misclassified table { display-mode: "form" }
def show_errors(gold, predictions):
    rows = [{"id": g["id"], "gold": g["label"], "pred": p, "text": g["text"]}
            for g, p in zip(gold, predictions) if g["label"] != p]
    print(f"{len(rows)} of {len(gold)} wrong.")
    return pd.DataFrame(rows)

In [ ]:
#@title 🔧 Library cell: save_predictions / load_predictions { display-mode: "form" }
def save_predictions(predictions, path):
    """Freeze a list of predicted labels to JSON."""
    with open(path, "w", encoding="utf-8") as f:
        json.dump(predictions, f)
    print(f"Saved {len(predictions)} predictions to {path}")

def load_predictions(url_or_path):
    """Read a frozen predictions list — a committed URL or a local path."""
    if str(url_or_path).startswith("http"):
        raw = urllib.request.urlopen(url_or_path).read().decode("utf-8")
        predictions = json.loads(raw)
    else:
        predictions = json.loads(open(url_or_path, encoding="utf-8").read())
    print(f"Loaded {len(predictions)} frozen predictions.")
    return predictions

In [ ]:
gold = load_gold(GOLD_URL)

### Iteration 0 — zero-shot   ✏️ YOU EDIT

Just describe the task. This is your baseline; note its macro-F1.

In [ ]:
PROMPT_ZERO = """You are an expert rater of English sentence difficulty using the CEFR scale.
Classify the sentence into exactly one of: A1, A2, B1, B2, C1, C2.
Answer with the level only.

Sentence: {text}"""

pred_zero = run_prompt(PROMPT_ZERO, gold)
evaluate(gold, pred_zero)

### Iteration 1 — few-shot   ✏️ YOU EDIT

Add a few **labeled examples** so the model can pattern-match. The examples are hand-written here (feel free to draw more from `cefr_pool.json` — never from the gold set you're scoring on, or you'd be testing on training data).

In [ ]:
PROMPT_FEWSHOT = """You are an expert rater of English sentence difficulty using the CEFR scale.
Classify the sentence into exactly one of: A1, A2, B1, B2, C1, C2. Answer with the level only.

Examples:
Sentence: "I have a cat." -> A1
Sentence: "She went to the shops because she needed some milk." -> A2
Sentence: "The results suggest a modest but consistent improvement." -> B2
Sentence: "Notwithstanding these caveats, the framework generalises well." -> C2

Sentence: {text}"""

pred_few = run_prompt(PROMPT_FEWSHOT, gold)
evaluate(gold, pred_few)

### Iteration 2 — chain-of-thought (CoT)   ✏️ YOU EDIT

Ask the model to **reason first, then answer**. Giving it room to think often helps on borderline items.

In [ ]:
PROMPT_COT = """You are an expert rater of English sentence difficulty using the CEFR scale.
Think step by step about the vocabulary and grammar, then decide the level.
Do NOT mention any other CEFR level while reasoning.
End your answer with the final level on its own, exactly one of: A1, A2, B1, B2, C1, C2.

Sentence: {text}"""

pred_cot = run_prompt(PROMPT_COT, gold)
evaluate(gold, pred_cot)

::: {.callout-note}
## A real limitation to notice
`run_prompt` grabs the *first* CEFR level it sees in the reply. With chain-of-thought the model may mention a level mid-reasoning, so the parser can pick the wrong one — which is exactly why the prompt says *don't mention other levels*. If CoT scores *worse* than few-shot, check `show_errors` to see whether it's the model or the parser.
:::

### Compare the three

Fill in the macro-F1 you saw at each step. This little table *is* your result.

| Iteration | macro-F1 |
|---|:--:|
| 0 · zero-shot | … |
| 1 · few-shot | … |
| 2 · chain-of-thought | … |

## Part B · Corpus Lab — your own prompt-iteration study

::: {.callout-warning}
## 🚧 TODO — to be written
This lab is still being written. When it is, you will:

1. Pick one class the model handles worst (use `show_errors`).
2. Write **one concrete prompt change** you predict will help, and test it.
3. Log each iteration's macro-F1 in the table below and explain the change.

| Iteration | Prompt change | macro-F1 |
|---|---|:--:|
| … | … | … |
:::

---
## ✅ Before you submit

1. **Runtime → Run all** and check every cell ran without error.
2. Tutorial outputs are visible (tables / charts / the model's answers).
3. Every Corpus Lab self-check prints ✅ (or your TODO answers are filled in).
4. **File → Download → Download `.ipynb`** and upload that one file.